## **Connect Colab**

In [ ]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "github"
os.environ["KAGGLE_USERNAME"] = "kaggle_username"
os.environ["KAGGLE_KEY"] = "kaggle_key"

In [ ]:
!dir /content/drive/MyDrive/apai/resnet50/models/

ResNet50_10C.pth


In [ ]:
!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 15 file(s) under /src/
src/cl_strategies/ewc.py -> /content/src/cl_strategies/ewc.py
src/cl_strategies/naive.py -> /content/src/cl_strategies/naive.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/meta_learning/mamal.py -> /content/src/meta_learning/mamal.py
src/meta_learning/reptil.py -> /content/src/meta_learning/reptil.py
src/models/baselines.py -> /content/src/models/baselines.py
src/models/pretrained.py -> /content/src/models/p

## **Download DATASET**

In [ ]:
!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh  # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
Dataset path: /kaggle/input/ucf101-action-recognition

Detecting path and updating config.py...
Found in Kaggle input: /kaggle/input/ucf101-action-recognition/
DATASET_ROOT = '/kaggle/input/ucf101-action-recognition/'
OUTPUT_ROOT  = '/kaggle/working/ucf101-processed/'

Setup complete!
config.py updated on Drive.
Restarting kernel in 5 seconds...
Dataset cache stays on disk.
Run Cell 3 after restart to verify.
Restarting in 5...
Restarting in 4...
Restarting in 3...
Restarting in 2...
Restarting in 1...


In [ ]:
# imports
%run /content/src/imports.py
criterion = nn.CrossEntropyLoss()

Using device: cuda


## **Preprocess the data**

In [ ]:
preprocess_dataset()



--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : /kaggle/working/ucf101-processed/
Classes: 10 | Limit: Full

Processing split: train



Processing split: val



Processing split: test



Done! Dataset ready at: /kaggle/working/ucf101-processed/


## **Create Loaders**

In [ ]:
class_to_idx_10 = {cls: i for i, cls in enumerate(SELECTED_CLASSES)}
print(class_to_idx_10)

TRAIN_DIR = os.path.join(OUTPUT_ROOT, "train")
VAL_DIR = os.path.join(OUTPUT_ROOT, "val")
TEST_DIR = os.path.join(OUTPUT_ROOT, "test")

train_dataset_10 = UCF101Clips(
    TRAIN_DIR,
    class_to_idx_10
)

val_dataset_10 = UCF101Clips(
    VAL_DIR,
    class_to_idx_10
)

test_dataset_10 = UCF101Clips(
    TEST_DIR,
    class_to_idx_10
)

train_loader_10 = DataLoader(train_dataset_10, batch_size=BATCH_SIZE, shuffle=True)
val_loader_10   = DataLoader(val_dataset_10, batch_size=BATCH_SIZE, shuffle=False)
test_loader_10  = DataLoader(test_dataset_10, batch_size=BATCH_SIZE, shuffle=False)

{'PlayingTabla': 0, 'PommelHorse': 1, 'JumpingJack': 2, 'PushUps': 3, 'PoleVault': 4, 'HorseRace': 5, 'HighJump': 6, 'Drumming': 7, 'HorseRiding': 8, 'Diving': 9}


## **Upload Model**

In [ ]:
from torchvision.models import mobilenet_v2
import torch.nn as nn
from torch.nn import functional as F

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2

class StudentModel(nn.Module):
    def __init__(self, num_classes=10, hidden_size=256, dropout_p=0.5):
        super().__init__()

        # CNN backbone (frame encoder)
        self.backbone = mobilenet_v2(weights='DEFAULT').features

        self.align = nn.Conv2d(1280, 2048, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # temporal model
        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_size, num_classes)

    # -----------------------------
    # FRAME ENCODER (NEW)
    # -----------------------------
    def encode_frame(self, frame):
        x = self.backbone(frame)
        x = self.align(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x

    # -----------------------------
    # FULL SEQUENCE FOR INFERENCE
    # (still exists but NOT used in training)
    # -----------------------------
    def forward(self, x):
        B, T, C, H, W = x.shape

        feats = []

        for t in range(T):
            frame = x[:, t]          # [B,3,224,224]
            f = self.encode_frame(frame)
            feats.append(f)

        feats = torch.stack(feats, dim=1)  # [B,T,2048]

        out, _ = self.lstm(feats)

        x = out[:, -1, :]
        logits = self.fc(self.dropout(x))

        return logits, x


    
student = StudentModel().to('cuda')

In [ ]:
class TeacherModel(nn.Module):
    """
    ResNet50 (frozen except layer4) + LSTM + Dropout classifier.

    Args:
        hidden_size : LSTM hidden state size         (default: 256)
        num_classes : output classes                 (default: num_classes)
        dropout_p   : dropout probability before fc  (default: DROPOUT_P)
                      0.4 for base/task1, lower to 0.3 for task2+

    Input : [B, T, C, H, W]
    Output: [B, num_classes]
    """
    def __init__(self, hidden_size=256, num_classes=num_classes, dropout_p=None):
        super().__init__()

        if dropout_p is None:
            dropout_p = DROPOUT_P

        resnet = models.resnet50(weights=ResNet50_Weights.DEFAULT)

        # freeze all, then unfreeze layer4 only
        for param in resnet.parameters():
            param.requires_grad = False
        for param in resnet.layer4.parameters():
            param.requires_grad = True

        self.resnet  = nn.Sequential(*list(resnet.children())[:-1])  # remove fc
        self.lstm    = nn.LSTM(input_size=2048, hidden_size=hidden_size, batch_first=True)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        features = self.resnet(x.view(B * T, C, H, W))  # [B*T, 2048, 1, 1]
        features = features.view(B, T, -1)               # [B, T, 2048]
        out, _  = self.lstm(features)                   # [B, T, hidden]
        feat = out[:, -1, :]          # (B, hidden)
        logits = self.fc(self.dropout(feat))     # last hidden state only
        return logits
    
    def hidden_representation(self, x):
        with torch.no_grad():
            B, T, C, H, W = x.shape
            features = self.resnet(x.view(B * T, C, H, W))  # [B*T, 2048, 1, 1]
            features = features.view(B, T, -1)               # [B, T, 2048]
            out, _  = self.lstm(features)                   # [B, T, hidden]
            feat = out[:, -1, :]          # (B, hidden)
        return feat 
    
teacher_model = TeacherModel(num_classes = len(cfg.SELECTED_CLASSES)).to(device)
state      = torch.load(cfg.RESNET50_PATH, map_location=device)
teacher_model.load_state_dict(state)


<All keys matched successfully>

## KD

In [ ]:

def evaluate_model_kd(model, dataloader, device):
    """
    Evaluate model on a dataloader.

    Args:
        model        : PyTorch model
        dataloader   : DataLoader
        device       : 'cuda' or 'cpu'

    Returns:
        (accuracy, avg_loss)
    """
    model.eval()
    all_preds, all_labels = [], []
    total_loss    = 0.0
    total_samples = 0
    criterion     = nn.CrossEntropyLoss()

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            output, _ = model(data)
            loss   = criterion(output, target)
            pred   = output.argmax(dim=1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            total_loss    += loss.item() * data.size(0)
            total_samples += data.size(0)

    accuracy = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / total_samples
    return accuracy, avg_loss


In [ ]:
# Function to train a model for one single epoch
# import transformer_engine.torch as te

def train_one_epoch_kd(student_model, teacher_model, train_loader, criterion, optimizer, hidden_rep_loss_weight, ce_loss_weight, device):
    """
    Handles the training loop for a single epoch.
    """
    student_model.train()
    teacher_model.eval()  # Teacher in eval mode, no gradients

    for param in teacher_model.parameters():
        param.requires_grad = False  # Ensure teacher is frozen

    running_loss, correct, total = 0.0, 0, 0

    cosine_loss = nn.CosineEmbeddingLoss()

    for clips, labels in tqdm(train_loader, desc="  Training", leave=False):

        clips, labels = clips.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.float16): 
            with torch.no_grad():
                teacher_hidden_rep = teacher_model.hidden_representation(clips)

            outputs_student, student_hidden_rep = student_model(clips)

            hidden_rep_loss = cosine_loss(student_hidden_rep, teacher_hidden_rep, target=torch.ones(clips.size(0)).to(device))

            output_loss = criterion(outputs_student, labels)
            loss = hidden_rep_loss * hidden_rep_loss_weight + output_loss * ce_loss_weight
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs_student.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total, running_loss / total



In [ ]:

def train_model_kd(
    teacher_model,
    student_model,
    train_loader,
    val_loader,
    num_epochs=5,
    lr=1e-4,
    device='cuda',
    criterion=None, 
    optimizer=None,
    save_path=None,
    hidden_rep_loss_weight=0.25,
    ce_loss_weight=0.75,
):
    # Setup
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    if optimizer is None:
        optimizer = optim.Adam(student_model.parameters(), lr=lr)

    history = {
        'train_losses': [], 'val_losses': [],
        'train_accs': [], 'val_accs': [],
        'best_val_acc': 0.0
    }

    student_model.to(device)
    teacher_model.to(device)

    for epoch in range(num_epochs):
        # 1. Train using our helper
        train_acc, train_loss = train_one_epoch_kd(student_model, teacher_model, train_loader, criterion, optimizer, hidden_rep_loss_weight, ce_loss_weight, device)
        torch.cuda.empty_cache()  # Clear cache after each epoch to manage VRAM
        
        # 2. Validate using your existing evaluate_model
        val_acc, val_loss = evaluate_model_kd(student_model, val_loader, device)
        torch.cuda.empty_cache()  # Clear cache after validation

        # 3. Update History
        history['train_losses'].append(train_loss)
        history['train_accs'].append(train_acc)
        history['val_losses'].append(val_loss)
        history['val_accs'].append(val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}] | "
              f"Train Acc: {train_acc:.4f} Loss: {train_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} Loss: {val_loss:.4f}")

        # 4. Save best model
        if save_path and val_acc > history['best_val_acc']:
            history['best_val_acc'] = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  --> Model saved to {save_path}")

    return history


In [ ]:


# INFERENCE
def test_model_kd(student_model, test_loader, device='cuda'):
    """
    Run inference on the test set without task offsets.

    Args:
        student_model : PyTorch student model
        test_loader  : DataLoader for the test set
        device       : 'cuda' or 'cpu'

    Returns:
        (test_accuracy, all_preds, all_labels)
    """
    student_model.eval()
    all_preds, all_labels = [], []

    print(f"Running Inference on Test Set...")
    with torch.no_grad():
        for clips, labels in tqdm(test_loader, desc="  Testing", leave=False):
            clips, labels = clips.to(device), labels.to(device)
            
            outputs, _ = student_model(clips)
            preds   = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = accuracy_score(all_labels, all_preds)
    print(f"Test Accuracy: {test_acc:.4f}")
    
    return test_acc, all_preds, all_labels


In [ ]:
kd_results = train_model_kd(
    teacher_model=teacher_model,
    student_model=student,
    train_loader=train_loader_10,
    val_loader=val_loader_10,
    # criterion=criterion,
    # optimizer=optimizer,
    hidden_rep_loss_weight= 0.35,
    ce_loss_weight=0.65,
    device=device
)


Epoch [1/5] | Train Acc: 0.1650 Loss: 1.9196 | Val Acc: 0.1879 Loss: 2.2203


Epoch [2/5] | Train Acc: 0.2002 Loss: 1.8040 | Val Acc: 0.1939 Loss: 2.0884


Epoch [3/5] | Train Acc: 0.2304 Loss: 1.7243 | Val Acc: 0.2545 Loss: 2.0106


Epoch [4/5] | Train Acc: 0.2797 Loss: 1.6669 | Val Acc: 0.2424 Loss: 2.0022


Epoch [5/5] | Train Acc: 0.2716 Loss: 1.6343 | Val Acc: 0.2061 Loss: 2.2260


In [ ]:
plot_training_results(kd_results)

In [ ]:
test_acc, all_preds, all_labels = test_model_kd(student_model, test_loader=test_loader_10, device=device)

In [ ]:
plot_confusion_matrix(all_labels, all_preds, num_classes=10, classes_list=SELECTED_CLASSES)